# HELPER FUNCTIONS *keeping the code clean

In [52]:
def create_target_horizons(price_5s, horizons=[5, 10, 15, 30, 45, 60], interval_seconds=5):
    """
    Create multi-horizon price change targets from a 5-second interval price dataframe.

    Parameters:
        price_5s (pd.DataFrame): DataFrame with 'price' column indexed by ts_event.
        horizons (list): List of prediction horizons in seconds.
        interval_seconds (int): Time interval of the price_5s DataFrame (default 5s).

    Returns:
        pd.DataFrame: target_horizons with columns for each horizon containing future price changes.
    """
    target_dict = {}
    
    # Compute shift in rows for each horizon
    rows_ahead = [h // interval_seconds for h in horizons]
    
    for h, n_rows in zip(horizons, rows_ahead):
        # Future price change = price at t+h minus current price
        target_dict[f"{h}s"] = price_5s['price'].shift(-n_rows) - price_5s['price']
    
    # Create DataFrame
    target_horizons = pd.DataFrame(target_dict, index=price_5s.index)
    
    # Drop rows with NaNs (last rows where future price doesn't exist)
    target_horizons = target_horizons.dropna()
    
    return target_horizons

### PROJECT BEGINS FROM THIS POINT ON

In [53]:
import pandas as pd

horizons = [5, 10, 15, 30, 45, 60]  # in seconds up to a minute

Below is preparing the target price data for the different horizons that is going to be run.

In [54]:

# import price data
df = pd.read_csv('first_25000_rows.csv')
price_df = df[['ts_event', 'price']]
price_df["ts_event"] = pd.to_datetime(price_df["ts_event"])
price_df.set_index("ts_event", inplace=True)

price_5s = price_df.resample('5s').first().reset_index()
price_5s = price_5s.dropna()
price_5s.set_index("ts_event", inplace=True)
print(price_5s)

                            price
ts_event                         
2024-10-21 11:54:25+00:00  233.62
2024-10-21 11:54:35+00:00  233.90
2024-10-21 11:54:40+00:00  233.67
2024-10-21 11:54:50+00:00  233.79
2024-10-21 11:54:55+00:00  233.48
...                           ...
2024-10-21 13:04:00+00:00  233.70
2024-10-21 13:04:05+00:00  233.68
2024-10-21 13:04:10+00:00  233.68
2024-10-21 13:04:15+00:00  233.54
2024-10-21 13:04:20+00:00  233.46

[643 rows x 1 columns]


C:\Users\soroc\AppData\Local\Temp\ipykernel_25228\2055662240.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  price_df["ts_event"] = pd.to_datetime(price_df["ts_event"])


In [55]:
target_horizons = create_target_horizons(price_5s, horizons)
print(target_horizons)

                             5s   10s   15s   30s   45s   60s
ts_event                                                     
2024-10-21 11:54:25+00:00  0.28  0.05  0.17 -0.03 -0.02  0.11
2024-10-21 11:54:35+00:00 -0.23 -0.11 -0.42 -0.39 -0.23 -0.42
2024-10-21 11:54:40+00:00  0.12 -0.19 -0.08  0.06 -0.12  0.33
2024-10-21 11:54:50+00:00 -0.31 -0.20 -0.20 -0.19 -0.06 -0.04
2024-10-21 11:54:55+00:00  0.11  0.11  0.03  0.19  0.00  0.29
...                         ...   ...   ...   ...   ...   ...
2024-10-21 13:02:50+00:00 -0.06 -0.22 -0.01  0.04 -0.19  0.07
2024-10-21 13:02:55+00:00 -0.16  0.05  0.04  0.02 -0.12  0.11
2024-10-21 13:03:00+00:00  0.21  0.20  0.14  0.24  0.13  0.27
2024-10-21 13:03:05+00:00 -0.01 -0.07  0.05 -0.18  0.08 -0.08
2024-10-21 13:03:10+00:00 -0.06  0.06 -0.02 -0.16  0.07 -0.15

[631 rows x 6 columns]


# Importing all features and targets dataframes

In [56]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# pre-processed target horizons
target_horizons

# OFI features
ofi_best_level = pd.read_csv("feature_outputs/ofi_best_level.csv", index_col="ts_event", parse_dates=True)
multi_level_ofi = pd.read_csv("feature_outputs/ofi_multi_level.csv", index_col="ts_event", parse_dates=True)
integrated_ofi = pd.read_csv("feature_outputs/ofi_integrated.csv", index_col="ts_event", parse_dates=True)

ofi_best_level.index = pd.to_datetime(ofi_best_level.index).tz_localize('UTC')
multi_level_ofi.index = pd.to_datetime(multi_level_ofi.index).tz_localize('UTC')
integrated_ofi.index = pd.to_datetime(integrated_ofi.index).tz_localize('UTC')

print("OFI Best Level:")
print(ofi_best_level.head())

print("\nMulti-Level OFI:")
print(multi_level_ofi.head())

print("\nIntegrated OFI:")
print(integrated_ofi.head())

print("\nTarget Horizons:")
print(target_horizons.head())

OFI Best Level:
                           best_level_ofi
ts_event                                 
2024-10-21 11:54:25+00:00             3.0
2024-10-21 11:54:30+00:00             0.0
2024-10-21 11:54:35+00:00           401.0
2024-10-21 11:54:40+00:00          -278.0
2024-10-21 11:54:45+00:00             0.0

Multi-Level OFI:
                           ofi_00  ofi_01  ofi_02  ofi_03  ofi_04  ofi_05  \
ts_event                                                                    
2024-10-21 11:54:25+00:00   100.0   100.0   100.0     0.0     0.0     0.0   
2024-10-21 11:54:30+00:00     0.0     0.0     0.0     0.0     0.0     0.0   
2024-10-21 11:54:35+00:00   -80.0   -80.0   -80.0   -80.0   -80.0   -80.0   
2024-10-21 11:54:40+00:00   120.0   120.0   120.0   120.0   120.0   120.0   
2024-10-21 11:54:45+00:00     0.0     0.0     0.0     0.0     0.0     0.0   

                           ofi_06  ofi_07  ofi_08  ofi_09  
ts_event                                                   
2024-10-21 1